In [ ]:
!pip install -q transformers datasets torch sentencepiece accelerate

In [ ]:
from datasets import load_dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split='train[:10]')

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_name = "facebook/nllb-200-distilled-600M"
device = "cuda" if torch.cuda.is_available() else "cpu"


tokenizer = AutoTokenizer.from_pretrained(model_name, src_lang="eng_Latn")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def translate_text(text, tgt_lang="mkd_Cyrl"):

    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)


    forced_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_id,
        max_length=512
    )

    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

print("Моделот е подготвен!")

In [ ]:
def process_batch(example):

    try:
        example["instruction_mk"] = translate_text(example["instruction"])
        example["response_mk"] = translate_text(example["response"])
    except Exception as e:
        print(f"Грешка кај еден ред: {e}")
        example["instruction_mk"] = "ГРЕШКА"
        example["response_mk"] = "ГРЕШКА"
    return example

print("Започнувам со преведување на 10 примери...")
translated_dataset = dataset.map(process_batch)
print("Готово!")

In [ ]:
import pandas as pd
df = translated_dataset.to_pandas()
display(df[['instruction', 'instruction_mk', 'response', 'response_mk']].head(20))

In [ ]:
!pip install -q evaluate sacrebleu sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util
import evaluate

In [ ]:
chrf = evaluate.load("chrf")

In [ ]:
sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
import pandas as pd
import evaluate
from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
print("Вчитувам модели за евалуација...")
chrf = evaluate.load("chrf")
# Користиме повеќејазичен модел кој ги мапира MK и EN во ист простор
sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
def run_evaluation(dataset_to_eval):
    # Функција за обработка на секој ред
    def process_eval(batch):
        # Пресметка на семантичка сличност за Инструкциите
        emb_eng_inst = sbert_model.encode(batch["instruction"], convert_to_tensor=True)
        emb_mk_inst = sbert_model.encode(batch["instruction_mk"], convert_to_tensor=True)
        batch["semantic_score_inst"] = util.cos_sim(emb_eng_inst, emb_mk_inst).diag().tolist()

        # Пресметка на семантичка сличност за Одговорите
        emb_eng_resp = sbert_model.encode(batch["response"], convert_to_tensor=True)
        emb_mk_resp = sbert_model.encode(batch["response_mk"], convert_to_tensor=True)
        batch["semantic_score_resp"] = util.cos_sim(emb_eng_resp, emb_mk_resp).diag().tolist()

        return batch

    print("Пресметувам семантичка сличност...")
    results = dataset_to_eval.map(process_eval, batched=True, batch_size=10)

    # Пресметка на chrF (покрива морфологија на MK јазик)
    print("Пресметувам chrF score...")
    chrf_results = chrf.compute(
        predictions=results["instruction_mk"],
        references=[[ref] for ref in results["instruction"]]
    )

    return results, chrf_results["score"]

In [ ]:
eval_results, final_chrf = run_evaluation(translated_dataset)

In [ ]:
df_results = eval_results.to_pandas()

In [ ]:
print("-" * 30)
print(f"ПРОСЕЧЕН chrF SCORE: {final_chrf:.2f}")
print(f"ПРОСЕЧНА СЛИЧНОСТ (Инструкции): {df_results['semantic_score_inst'].mean():.4f}")
print(f"ПРОСЕЧНА СЛИЧНОСТ (Одговори): {df_results['semantic_score_resp'].mean():.4f}")
print("-" * 30)

In [ ]:
df_results.to_csv("evaluated_translations.csv", index=False)
print("Резултатите се зачувани во 'evaluated_translations.csv'!")

In [ ]:
print("\nНајдобри преводи според модел:")
display(df_results.nlargest(5, 'semantic_score_inst')[['instruction', 'instruction_mk', 'semantic_score_inst']])